# Ketakutan dan Musim pada Sistem Predator–Mangsa

**ID proyek:** `O005-LEGA-V101-PRJ09`  
**Status:** titik awal pedagogis yang ditulis secara independen.

Notebook ini menggunakan data sintetis/terbuka saja. Notebook ini **bukan** kode atau data dari makalah yang dikutip dalam bab sumber dan **bukan** klaim reproduksi hasil penelitian mana pun.


## Pertanyaan pemodelan

Bagaimana respons antipredator dan pertumbuhan musiman bersama-sama mengubah amplitudo populasi?

Tujuan kerja: tetapkan sistem, jalankan eksperimen deterministik, periksa invarian, visualisasikan perilaku, lalu kritik kecukupan model.


In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

SEED = 2026082209
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=6, suppress=True)


## Struktur dan asumsi

Ketakutan mengurangi laju pertumbuhan efektif mangsa melalui faktor rasional; pertumbuhan intrinsik berosilasi sinusoidal; predasi bertipe bilinear.

Semua skala dan parameter di notebook ini bersifat ilustratif. Ubah satu asumsi pada satu waktu dan catat dampaknya pada keluaran serta invarian.


In [ ]:
r0, season_amp, period, K = 0.75, 0.28, 24.0, 3.0
attack, conversion, mortality = 0.42, 0.32, 0.26
t_eval = np.linspace(0.0, 144.0, 1153)

def seasonal_growth(t):
    return r0 * (1.0 + season_amp * np.sin(2.0 * np.pi * t / period))

def fear_run(fear):
    def rhs(t, y):
        prey, predator = y
        growth = seasonal_growth(t) * prey * (1.0 - prey / K) / (1.0 + fear * predator)
        loss = attack * prey * predator
        return [growth - loss, conversion * loss - mortality * predator]
    return solve_ivp(rhs, (0.0, 144.0), [1.6, 0.55], t_eval=t_eval, rtol=1e-9, atol=1e-11)

fear_runs = {0.0: fear_run(0.0), 0.9: fear_run(0.9)}
forcing = seasonal_growth(t_eval)


## Pemeriksaan numerik

Pemeriksaan berikut sengaja berada di dalam notebook: eksekusi berhenti bila suatu invarian dasar gagal. Ini bukan bukti bahwa model benar; ini hanya bukti bahwa implementasi memenuhi kontrak numerik terbatasnya.


In [ ]:
for run in fear_runs.values():
    assert run.success and np.isfinite(run.y).all() and np.min(run.y) > 0.0
np.testing.assert_allclose(forcing.max() - forcing.min(), 2.0 * r0 * season_amp, rtol=2e-4)
assert np.max(np.abs(fear_runs[0.0].y - fear_runs[0.9].y)) > 0.10


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
for fear, run in fear_runs.items():
    axes[0].plot(t_eval, run.y[0], label=f"mangsa, f={fear}")
    axes[1].plot(t_eval, run.y[1], label=f"predator, f={fear}")
axes[0].plot(t_eval, forcing, color="gray", alpha=0.5, linestyle=":", label="pertumbuhan musiman")
axes[0].set(ylabel="mangsa / pemaksaan", title="Ketakutan dan pemaksaan musiman")
axes[1].set(xlabel="waktu", ylabel="predator")
axes[0].legend(fontsize=8)
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()
plt.close(fig)


## Validasi, identifikasi, dan keterbatasan

Keterbatasan awal: Indeks ketakutan tidak diukur langsung, musim tunggal terlalu sederhana, dan model tidak memuat perlindungan habitat atau keterlambatan reproduksi.

Jawab sebelum menafsirkan gambar:

1. Besaran apa yang benar-benar dapat diamati, dan bagaimana galat pengukurannya dimodelkan?
2. Parameter mana yang dapat diidentifikasi dari keluaran tersebut? Tunjukkan dengan profil galat, pemisahan latih/uji, atau eksperimen sensitivitas.
3. Invarian atau pola kualitatif apa yang harus tetap benar ketika ukuran langkah, benih acak, atau resolusi diubah?
4. Temukan satu skenario kegagalan model dan jelaskan data tambahan yang diperlukan untuk membedakannya dari model alternatif.


## Daftar periksa reproduksibilitas

- [ ] Gunakan CPython dan versi paket tepat seperti `requirements.lock`.
- [ ] Jalankan ulang dari kernel kosong tanpa jaringan.
- [ ] Pertahankan nilai `SEED` (benih acak), lalu ulangi dengan sedikitnya lima benih acak lain dan laporkan variasinya.
- [ ] Catat setiap perubahan parameter, persamaan, toleransi, serta pembagian data.
- [ ] Pastikan semua uji lulus dan jelaskan mengapa tiap uji relevan.
- [ ] Simpan hasil turunan di luar notebook sumber; notebook distribusi harus tetap tanpa keluaran tersimpan.
- [ ] Bedakan hasil simulasi, data sintetis, dan klaim empiris secara eksplisit.
